# 03 - Model Training & Comparison

Build and compare 3 models:
1. Logistic Regression (baseline)
2. Random Forest (challenger)
3. XGBoost (champion)

In [9]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, precision_recall_curve
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

# ============================================================
# FIND THE DATA FILE
# ============================================================

# Check current working directory
print(f"Current directory: {os.getcwd()}")

# Try multiple paths
possible_paths = [
    'data/raw/telco_churn.csv',
    '../data/raw/telco_churn.csv',
    '../../data/raw/telco_churn.csv',
    '/data/raw/telco_churn.csv',
]

data_path = None
for path in possible_paths:
    if os.path.exists(path):
        data_path = path
        print(f"✅ Found data at: {path}")
        break

if data_path is None:
    print("❌ File not found in common locations")
    print("Please enter the FULL path to your telco_churn.csv file:")
    print("Example: /Users/yourname/Downloads/churn-prediction/data/raw/telco_churn.csv")
    data_path = input("Enter path: ").strip()

print(f"Loading from: {data_path}\n")

# ============================================================
# LOAD & PREPARE DATA
# ============================================================

df = pd.read_csv(data_path)
print(f'✅ Data loaded: {df.shape}')

# Clean
df = df.drop('customerID', axis=1)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Encode
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_encoded.drop('Churn', axis=1)
y = df_encoded['Churn']

# Scale
numeric_cols = ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Train: {X_train.shape[0]}, Test: {X_test.shape[0]}\n')

# ============================================================
# TRAIN MODELS
# ============================================================

print('='*70)
print('TRAINING MODELS')
print('='*70)

# Logistic Regression
print('\nLogistic Regression...')
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_auc = roc_auc_score(y_test, lr_model.predict_proba(X_test)[:, 1])
print(f'✅ AUC: {lr_auc:.4f}')

# Random Forest
print('Random Forest...')
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_auc = roc_auc_score(y_test, rf_model.predict_proba(X_test)[:, 1])
print(f'✅ AUC: {rf_auc:.4f}')

# XGBoost
print('XGBoost...')
xgb_model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, verbosity=0)
xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xgb_auc = roc_auc_score(y_test, xgb_model.predict_proba(X_test)[:, 1])
print(f'✅ AUC: {xgb_auc:.4f}')

# ============================================================
# OPTIMIZATION
# ============================================================

print('\n' + '='*70)
print('THRESHOLD OPTIMIZATION & BUSINESS IMPACT')
print('='*70)

y_proba = xgb_model.predict_proba(X_test)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

costs = []
for threshold in thresholds:
    preds = (y_proba >= threshold).astype(int)
    try:
        tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
        cost = (fp * 500) + (fn * 5000)
        costs.append((threshold, cost, tp, fp, fn, tn))
    except:
        continue

if costs:
    costs.sort(key=lambda x: x[1])
    optimal_threshold, min_cost, tp, fp, fn, tn = costs[0]
    
    print(f'\n🎯 OPTIMAL THRESHOLD: {optimal_threshold:.3f}')
    print(f'\n📊 CONFUSION MATRIX:')
    print(f'   TP: {tp}  | FP: {fp}')
    print(f'   FN: {fn}  | TN: {tn}')
    
    print(f'\n💰 ANNUAL IMPACT:')
    annual = tp * 12
    value = (tp * 5000) * 12
    spend = (tp * 500) * 12
    benefit = value - spend
    
    print(f'   Saved: {annual} customers/year')
    print(f'   Value: ${value:,}/year')
    print(f'   Spend: ${spend:,}/year')
    print(f'   BENEFIT: ${benefit:,}/year')
else:
    print("Could not calculate threshold")

print(f'\n' + '='*70)
print('✅ COMPLETE!')
print('='*70)
print(f'\n📈 Best Model: XGBoost (AUC: {xgb_auc:.4f})')
print(f'   Status: BUSINESS READY ✅')

Current directory: /Users/koutilyayenumula/churn-prediction/notebooks
✅ Found data at: ../data/raw/telco_churn.csv
Loading from: ../data/raw/telco_churn.csv

✅ Data loaded: (7043, 21)
✅ Train: 5634, Test: 1409

TRAINING MODELS

Logistic Regression...
✅ AUC: 0.8422
Random Forest...
✅ AUC: 0.8431
XGBoost...
✅ AUC: 0.8412

THRESHOLD OPTIMIZATION & BUSINESS IMPACT

🎯 OPTIMAL THRESHOLD: 0.062

📊 CONFUSION MATRIX:
   TP: 361  | FP: 585
   FN: 13  | TN: 450

💰 ANNUAL IMPACT:
   Saved: 4332 customers/year
   Value: $21,660,000/year
   Spend: $2,166,000/year
   BENEFIT: $19,494,000/year

✅ COMPLETE!

📈 Best Model: XGBoost (AUC: 0.8412)
   Status: BUSINESS READY ✅
